# **Supervised Models**

## Preparing data

In [10]:
# Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [3]:
# Reading dataset + setting X and y
df = pd.read_csv('phishing_processed.csv')
df.head()

X = df.drop('result', axis=1)
y = df['result']

In [4]:
# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
# 5-fold cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = ['accuracy', 'precision', 'recall', 'f1']

def report_cv(model, X_train, y_train, name):
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    print(f"{name} CV Results:")
    for metric in scoring:
      mean = scores[f'test_{metric}'].mean()
      std = scores[f'test_{metric}'].std()
      print(f"{metric}: {mean:.4f} +/- {std:.4f}")

## Baseline: Logistic Regression

In [6]:
# Baseline model: Logistic Regression (with pipeline)
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print(classification_report(y_test, y_pred_lr))
report_cv(lr_pipeline, X_train, y_train, name="Logistic Regression")

              precision    recall  f1-score   support

           0       0.93      0.94      0.93      1255
           1       0.92      0.90      0.91       956

    accuracy                           0.92      2211
   macro avg       0.92      0.92      0.92      2211
weighted avg       0.92      0.92      0.92      2211

Logistic Regression CV Results:
accuracy: 0.9276 +/- 0.0055
precision: 0.9276 +/- 0.0103
recall: 0.9087 +/- 0.0072
f1: 0.9180 +/- 0.0061


## Decision Tree

In [7]:
# Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

print(classification_report(y_test, y_pred_dt))
report_cv(dt_model, X_train, y_train, name="Decision Tree")

              precision    recall  f1-score   support

           0       0.96      0.97      0.96      1255
           1       0.96      0.94      0.95       956

    accuracy                           0.96      2211
   macro avg       0.96      0.96      0.96      2211
weighted avg       0.96      0.96      0.96      2211

Decision Tree CV Results:
accuracy: 0.9603 +/- 0.0032
precision: 0.9603 +/- 0.0058
recall: 0.9503 +/- 0.0087
f1: 0.9552 +/- 0.0037


## Random Forest

In [8]:
# Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print(classification_report(y_test, y_pred_rf))
report_cv(rf_model, X_train, y_train, name="Random Forest")

              precision    recall  f1-score   support

           0       0.96      0.98      0.97      1255
           1       0.97      0.95      0.96       956

    accuracy                           0.97      2211
   macro avg       0.97      0.97      0.97      2211
weighted avg       0.97      0.97      0.97      2211

Random Forest CV Results:
accuracy: 0.9687 +/- 0.0037
precision: 0.9723 +/- 0.0080
recall: 0.9571 +/- 0.0066
f1: 0.9646 +/- 0.0041


## Model Comparison

In [11]:
# Comparing evaluation metrics between all models
results = []

models = {
    "Logistic Regression": y_pred_lr,
    "Decision Tree": y_pred_dt,
    "Random Forest": y_pred_rf
}

for name, preds in models.items():
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    results.append({
        "Model": name,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

results_df = pd.DataFrame(results)
results_df

,Model,Precision,Recall,F1-score
0,Logistic Regression,0.919235,0.904812,0.911966
1,Decision Tree,0.955556,0.944561,0.950026
2,Random Forest,0.972193,0.950837,0.961396
